In [1]:
import pandas as pd

## New Topic Classification

In [2]:
train_df = pd.read_csv('/content/drive/MyDrive/MLDatasets/text_classification/train_10k.csv',names=['label','heading','text'])
train_df.head()

,label,heading,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [3]:
train_df.label.value_counts()

,count
label,
3,10000
4,10000
2,10000
1,10000


In [4]:
label_encoding = {'1':'world','2':'sports','3':'business','4':'science_tech'}

## Build the vocabulary

In [7]:
import re
_token_reg = re.compile(r"[a-z0-9]+")
# this regex will tokenize the text into alphanumeric tokens


def tokenize(text):
  text=text.lower()
  tokens = _token_reg.findall(text)
  return tokens


In [9]:
print(train_df.iloc[0].text)

tokens = tokenize(train_df.iloc[0].text)
print(tokens)

Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
['reuters', 'short', 'sellers', 'wall', 'street', 's', 'dwindling', 'band', 'of', 'ultra', 'cynics', 'are', 'seeing', 'green', 'again']


In [10]:
from collections import Counter

def build_vocab(texts,max_size=50000):
  counter = Counter()
  for text in texts:
    tokens = tokenize(text)
    counter.update(tokens)

  return counter

# counter takes all tokens of corpus as input and computes their frequencies
# it counts how many times a given token has appeared in given corpus

In [11]:
counter = build_vocab(train_df['text'].to_list())

In [ ]:
counter

Counter({'reuters': 5703,
         'short': 236,
         'sellers': 15,
         'wall': 421,
         'street': 491,
         's': 17102,
         'dwindling': 12,
         'band': 48,
         'of': 30335,
         'ultra': 22,
         'cynics': 2,
         'are': 3054,
         'seeing': 39,
         'green': 182,
         'again': 332,
         'private': 186,
         'investment': 282,
         'firm': 514,
         'carlyle': 1,
         'group': 1361,
         'which': 1559,
         'has': 5881,
         'a': 36275,
         'reputation': 35,
         'for': 12770,
         'making': 404,
         'well': 388,
         'timed': 9,
         'and': 22575,
         'occasionally': 5,
         'controversial': 124,
         'plays': 48,
         'in': 27057,
         'the': 66859,
         'defense': 286,
         'industry': 613,
         'quietly': 39,
         'placed': 80,
         'its': 7182,
         'bets': 8,
         'on': 16223,
         'another': 714,
         'part

In [12]:
len(counter)

38029

In [ ]:
len(counter.most_common(30000))
## most common method helps to filter the most occuring tokens for building vocab

30000

In [14]:
class Vocab:
  def __init__(self,counter,max_vocab_size):
    limited_counter = counter.most_common(max_vocab_size)

    self.idxTotoken=[]
    for token,freq in limited_counter:
      self.idxTotoken.append(token)

    self.idxTotoken.append('<pad>')
    self.idxTotoken.append('<unk>')

    self.tokenToidx = {}

    for idx,token in enumerate(self.idxTotoken):
      self.tokenToidx[token]=idx

    self.pad_token_idx = self.tokenToidx['<pad>']
    self.unk_token_idx = self.tokenToidx['<unk>']



  def encode(self,tokens):
    token_ids = []
    for token in tokens:
      token_id = self.tokenToidx.get(token,self.unk_token_idx)
      # if token in not part of vocab - use unk token_id for that token
      token_ids.append(token_id)

    return token_ids


In [15]:
vocab = Vocab(counter=counter,max_vocab_size=30000)

In [16]:
text = 'Paris is capital of France'
tokens = tokenize(text)
print(tokens)

['paris', 'is', 'capital', 'of', 'france']


In [17]:
token_ids = vocab.encode(tokens)
print(token_ids)

[800, 17, 406, 3, 770]


In [23]:
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence
import torch

In [37]:
class TextDataset(Dataset):
  def __init__(self,texts,labels,vocab,max_len=256):
    self.vocab=vocab
    self.max_len=max_len
    self.datapoints = []
    for text,label in zip(texts,labels):
      tokens = tokenize(text)[:self.max_len] #truncation
      token_ids = self.vocab.encode(tokens)
      self.datapoints.append((torch.tensor(token_ids,dtype=torch.long),label))

  def __len__(self):
    return len(self.datapoints)

  # this function is needed for working with indexing
  def __getitem__(self,idx):
    return self.datapoints[idx]



# applying padding for batches
# converting data into torch tensors
def collate_fn(pad_index:int):
  def collate(batch):
    sequences,labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences],dtype=torch.long)
    padded = pad_sequence(sequences,batch_first=True,padding_value=pad_index)
    labels = torch.tensor(labels,dtype=torch.long)
    return padded,lengths,labels
  return collate


# torch.tensor -> matrix datatypes in torch
# We are converting all matrixes in torch.long -> which is a float datatype in torch
# All matrixes need to be in same datatype for correct operations



## Why padding is needed ?
'''
text 1 : This is a great place
-> this is a great place
token_ids : t0 t1 t2 t3 t4

--padding
text 2 : Hello all
-> hello all <pad> <pad> <pad>
token_ids : t0 t1 t2 t3 t4


--truncation
text 3 : There is a very big waterfall
-> there is a very big
token_ids : t0 t1 t2 t3 t4

Now as the number of token ids are matching, these two texts can be passed in a single batch matrix
'''


'\ntext 1 : This is a great place\n-> this is a great place\ntoken_ids : t0 t1 t2 t3 t4\n\n--padding\ntext 2 : Hello all \n-> hello all <pad> <pad> <pad>\ntoken_ids : t0 t1 t2 t3 t4\n\n\n--truncation\ntext 3 : There is a very big waterfall\n-> there is a very big\ntoken_ids : t0 t1 t2 t3 t4\n\nNow as the number of token ids are matching, these two texts can be passed in a single batch matrix\n'

In [27]:
vocab.unk_token_idx

30001

## RNN Model Architecture

In [29]:
from torch import nn

class RNNTextClf(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim,num_classes,pad_idx,num_layers=1,
               dropout=0.3):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=pad_idx)
    self.lstm = nn.LSTM(
        input_size = embed_dim,
        hidden_size = hidden_dim, # number of nodes in LSTM layer
        batch_first = True,
        bidirectional = True,
        num_layers = num_layers,
        dropout = dropout if num_layers>1 else 0.0
    )
    self.dropout = nn.Dropout(dropout)

    # classification head - output layer
    self.clf_head = nn.Linear(in_features= hidden_dim*2, # *2 is due to bidirectional nature
                              out_features=num_classes)


  def forward(self,batch,lengths):
    # batch : (batch_size,seq_lengths)
    embedded = self.embedding(batch)
    embedded = self.dropout(embedded)

    packed = pack_padded_sequence(
        embedded,lengths,batch_first=True,enforce_sorted=False
    )
    _,(hidden,_) = self.lstm(packed)

    # hidden : num_layers*2(bidirectional) x batch_size x hidden_dim
    # hidden[-2] : left to right context vector
    # hidden[-1] : right to left context vector
    concat_hidden = torch.concat((hidden[-2],hidden[-1]),dim=1)
    concate_hidden = self.dropout(concat_hidden)
    return self.clf_head(concat_hidden)


In [35]:
criterion = nn.CrossEntropyLoss()
## this is two in one loss function , which applies softmax on given inputs and then computes loss wrt labels
## Due to this we don't need a dedicated softmax activation in last layer of model

def run_epoch(model,loader,criterion,device,optimizer):
  model.train()
  total_loss = 0
  with torch.set_grad_enabled(True):
    for i ,(padded,lengths,labels) in enumerate(loader):
      output = model(padded,lengths)
      loss = criterion(output,labels)
      # CELoss will first apply softmax on output and then compute loss wrt labels

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_loss+=loss.item()

      print(f"Batch : {i}\n Batch Loss : {loss.item()}\n Total Epoch Loss : {total_loss}")

In [36]:
test_df = pd.read_csv('/content/drive/MyDrive/MLDatasets/text_classification/test_2k.csv',names=['label','heading','text'])
test_df.head()

,label,heading,text
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [39]:
train_df.isna().sum()

,0
label,0
heading,0
text,0


In [42]:
train_df['text']=train_df['text'].apply(lambda x:x.strip())
train_df[train_df['text']=='']

,label,heading,text


In [43]:
train_texts = train_df['text'].to_list()
train_labels = train_df['label'].to_list()

In [44]:
test_df['text']=test_df['text'].apply(lambda x:x.strip())
test_df[test_df['text']=='']

,label,heading,text


In [45]:
test_texts = test_df['text'].to_list()
test_labels = test_df['label'].to_list()

In [51]:
## Create datasets
MAX_LEN=256
train_ds = TextDataset(train_texts,train_labels,vocab,max_len=MAX_LEN)
test_ds = TextDataset(test_texts,test_labels,vocab,max_len=MAX_LEN)

collate = collate_fn(vocab.pad_token_idx)

In [48]:
train_ds[100]

(tensor([  298,   101,     1,  3714,   513,  1214,     2,  8461,  6214,     5,
         19384,   222,  1655,     2,    30,   167,     2,  3059,   708,     0,
         19385,  3060,    34,  1727,    57,  7024,     5,    89,    30,    12,
          2066,    12, 24457,    19,  9482,  3404]),
 4)

In [52]:
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,shuffle =True,collate_fn = collate)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle =True,collate_fn = collate)

In [67]:
batch = next(iter(train_loader))

In [68]:
# token_ids
batch[0].shape

torch.Size([64, 79])

In [60]:
batch[0]

tensor([[   35,    16,    25,  ..., 30000, 30000, 30000],
        [   16,   115,    30,  ..., 30000, 30000, 30000],
        [  130,     0,  1813,  ..., 30000, 30000, 30000],
        ...,
        [    1, 23823,   468,  ..., 30000, 30000, 30000],
        [  200,   301,    22,  ..., 30000, 30000, 30000],
        [ 1431,  1155,   659,  ..., 30000, 30000, 30000]])

In [58]:
# labels
batch[1].shape

torch.Size([64])